# Practice 53 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Employee contracts

5 employees, each with a different contract length. Answer these questions — no prescribed steps.

1. Add `contract_end`: hire date + contract length. Use `DateOffset(months=n)` per row.
2. Add `review_date`: 2 weeks before `contract_end`.
3. Add `days_remaining`: days from `2024-04-20` to `contract_end` — negative means expired.
4. Which department has the highest mean `days_remaining`? Use NumPy.
5. Who expires soonest after today (i.e. smallest positive `days_remaining`)?

In [ ]:
staff = pd.DataFrame({
    'name':             ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':             ['Eng', 'Sales', 'Eng', 'HR', 'Sales'],
    'hire_date':        pd.to_datetime(['2022-03-15', '2021-07-01', '2023-01-20',
                                        '2022-11-05', '2023-06-30']),
    'contract_months':  [18, 36, 12, 24, 12],
})

# Your code here

staff['contract_end'] = staff.apply(lambda row: row['hire_date'] + pd.DateOffset(months = row['contract_months']), axis = 1)
staff['review_date'] = staff['contract_end'] - pd.DateOffset(weeks = 2)
staff['days_remaining'] = (pd.to_datetime('2024-04-20') - staff['contract_end']).dt.days
m = staff.groupby('dept')['days_remaining'].mean()
m.index[np.argmax(m)]

pos = staff.loc[staff['days_remaining']>0].reset_index()
pos['name'][np.argmin(pos['days_remaining'])]


# pos1 = staff.loc[staff['days_remaining']>0]
# pos1.iloc[np.argmin(pos['days_remaining'])]['name']

'Carol'

---
## Level 2 — Year-over-year revenue

Monthly revenue for a store, Jan 2023 – Apr 2024. `month` is the index.

`shift(12, freq='MS')` shifts the **index** forward 12 months — values from 2023 land on the matching 2024 dates, so a join aligns last year's revenue with this year's.

- Join the shifted series back as `last_year`.
- `yoy_change`: revenue − last_year.
- `yoy_pct`: percentage growth, rounded to 1 decimal.
- `rolling_yoy`: 3-month rolling mean of `yoy_pct`.
- Which month had the steepest YoY decline? Print the date and the value.

In [45]:
monthly = pd.DataFrame(
    {'revenue': [125000, 118000, 132000, 141000, 155000, 162000,
                 158000, 171000, 149000, 138000, 143000, 168000,
                 142000, 131000, 148000, 159000]},
    index=pd.date_range('2023-01-01', periods=16, freq='MS'),
)
monthly.index.name = 'month'

# Your code here

s = monthly['revenue'].shift(12, freq = 'MS')

ss = monthly.join(s.rename('last_year'))
ss['yoy_change'] = ss['revenue'] - ss['last_year']
ss['yoy_pct'] = round(ss['yoy_change']/ss['last_year']*100,1)

ss['rolling_yoy'] = ss['yoy_pct'].rolling(3).mean()

print(ss['yoy_pct'].min(),ss['yoy_pct'].idxmin())


11.0 2024-02-01 00:00:00


---
## Level 3 — Tipping patterns (tips dataset)

Load the seaborn `tips` dataset. No steps — answer these four questions however you like.

1. Add `tip_pct`: tip as a percentage of total_bill, rounded to 1 decimal.
2. What is the mean and std of `tip_pct` for each combination of `sex` and `smoker`? Use NumPy inside a groupby.
3. On which (`day`, `time`) combination is the median `tip_pct` highest? (Use a pivot table or groupby — your call.)
4. Use a list comprehension to collect the `tip_pct` values for tables of 5 or more people. What is their mean tip percentage?

In [63]:
tips = sns.load_dataset('tips')

# Your code here
tips['tip_pct'] = round(tips['tip']/tips['total_bill']*100,1)
tips.groupby(['sex','smoker'], observed=True)['tip_pct'].agg(['mean','std'])

p = tips.pivot_table(
    index = 'day',
    columns= 'time',
    values='tip_pct',
    observed=True
)
p.stack().idxmax()

pct_5above = [pct5 for pct5, si in zip(tips['tip_pct'], tips['size']) if si>=5 ]

print(f'Mean pct is {np.mean(pct_5above):.2f}')

Mean pct is 14.81


In [58]:
tips

,total_bill,tip,sex,smoker,day,time,size,tip_pct
0,16.99,1.01,Female,No,Sun,Dinner,2,5.9
1,10.34,1.66,Male,No,Sun,Dinner,3,16.1
2,21.01,3.50,Male,No,Sun,Dinner,3,16.7
3,23.68,3.31,Male,No,Sun,Dinner,2,14.0
4,24.59,3.61,Female,No,Sun,Dinner,4,14.7
...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,20.4
240,27.18,2.00,Female,Yes,Sat,Dinner,2,7.4
241,22.67,2.00,Male,Yes,Sat,Dinner,2,8.8
242,17.82,1.75,Male,No,Sat,Dinner,2,9.8
